# ML Strategy Optimization Example

This notebook demonstrates indicator + model optimization using `trade_lab.ml_optimization.MLOptimizer`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'examples' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from trade_lab.backtesting import BacktestEngine
from trade_lab.indicators import EMA, RSI
from trade_lab.ml_optimization import IndicatorSpec, MLOptimizer, ModelPruner

## 1) Fetch data and split train/validation/test

In [ ]:
data_engine = BacktestEngine(ticker='SPY', start='2015-01-01', end='2025-01-01')
full_df = data_engine.fetch_data()

train_df = full_df[:'2020-12-31']
val_df = full_df['2021-01-01':'2022-12-31']
test_df = full_df['2023-01-01':]

print('Train:', len(train_df), 'Val:', len(val_df), 'Test:', len(test_df))

## 2) Define indicator search space

In [ ]:
indicator_specs = [
    IndicatorSpec(
        name='ema',
        indicator_class=EMA,
        period_low=5,
        period_high=60,
        lag_low=0,
        lag_high=10,
        max_lags=3,
        optional=False,
    ),
    IndicatorSpec(
        name='rsi',
        indicator_class=RSI,
        period_low=7,
        period_high=30,
        lag_low=0,
        lag_high=5,
        max_lags=2,
        optional=True,
    ),
]

## 3) Define model factory

In [ ]:
def model_factory(n_features: int):
    import keras

    model = keras.Sequential([
        keras.layers.Dense(64, activation='relu', input_shape=(n_features,)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1, activation='tanh'),
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

## 4) Run ML optimization

In [ ]:
ml_optimizer = MLOptimizer(
    indicator_specs=indicator_specs,
    model_factory=model_factory,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    metric='sharpe_ratio',
    n_trials=20,
    n_epochs=10,
    n_jobs=1,
)

ml_result = ml_optimizer.optimize()
print(ml_result.summary())

In [ ]:
ml_result.trials_df.sort_values('value', ascending=False).head(10)

## 5) Optional pruning

In [ ]:
pruner = ModelPruner(percentile=20)
pruned_result, report = pruner.prune_result(ml_result, fine_tune_epochs=5)

print('Zeroed fraction:', report['zero_fraction'])
print('Dead features:', report['dead_features'])
print(pruned_result.summary())